In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# 벡터 DB : Chroma vs Pinecone
- Chroma : 인메모리 vector DB, 로컬메모리 vector DB
- Pinecone : 클라우드 vector DB
    (Pinecone console에 api key 생성 -> .env) (PINECONE_API_KEY등록)

# 0. 패키지 설치

In [1]:
%pip install pinecone-client langchain-pinecone

  Using cached wrapt-1.17.2-cp310-cp310-win_amd64.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/587.6 kB ? eta -:--:--
   ---------------------------------------- 587.6/587.6 kB 6.9 MB/s eta 0:00:00
Using cached wrapt-1.17.2-cp310-cp310-win_amd64.whl (38 kB)

   ------ ---------------------------------  3/19 [pinecone-plugin-interface]
   ---------- -----------------------------  5/19 [pytest]
   ---------- -----------------------------  5/19 [pytest]
   ---------- -----------------------------  5/19 [pytest]
   ------------ ---------------------------  6/19 [pinecone-plugin-assistant]
   ------------ ---------------------------  6/19 [pinecone-plugin-assistant]
   ---------------- -----------------------  8/19 [vcrpy]
   ------------------ ---------------------  9/19 [syrupy]
   ------------------------- -------------- 12/19 [pytest-benchmark]
   --------------------------- ------------ 13/19 [pytest-asyncio]
   ----------------------------- ---------- 14/19 [p

# 1. Knowledge Base구성을 위한 데이터 생성

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docs/소득세법(법률)(제20615호)(20250701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter=text_splitter)

In [3]:
len(document_list)

181

In [1]:
# embedding = openAI API text-embedding-3-large
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [2]:
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 업로드할 때
index_name = "tax-index"
# database = PineconeVectorStore.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     index_name=index_name,
# )
# 업로드한 벡터DB 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name,
)

c:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. 답변생성을 위한 Retrieval

In [3]:
query = "연봉 5000만원인 직장인의 소득세는 얼마인가요?"
retriever = database.as_retriever(
)
retriever.invoke(query)

[Document(id='36e6c993-0d0a-4399-8e6b-9d3a7fb2de1b', metadata={'source': 'tax_docs/소득세법(법률)(제20615호)(20250701).docx'}, page_content='바. 「문화유산의 보존 및 활용에 관한 법률」에 따라 국가지정문화유산으로 지정된 서화ㆍ골동품의 양도로 발생하는 소득\n\n사. 서화ㆍ골동품을 박물관 또는 미술관에 양도함으로써 발생하는 소득\n\n아. 제21조제1항제26호에 따른 종교인소득 중 다음의 어느 하나에 해당하는 소득\n\n\u3000\u3000\u3000\u30001) 「통계법」 제22조에 따라 통계청장이 고시하는 한국표준직업분류에 따른 종교관련종사자(이하 “종교관련종사자”라 한다)가 받는 대통령령으로 정하는 학자금\n\n\u3000\u3000\u3000\u30002) 종교관련종사자가 받는 대통령령으로 정하는 식사 또는 식사대\n\n\u3000\u3000\u3000\u30003) 종교관련종사자가 받는 대통령령으로 정하는 실비변상적 성질의 지급액\n\n\u3000\u3000\u3000\u30004) 종교관련종사자 또는 그 배우자의 출산이나 6세 이하(해당 과세기간 개시일을 기준으로 판단한다) 자녀의 보육과 관련하여 종교단체로부터 받는 금액으로서 월 20만원 이내의 금액\n\n\u3000\u3000\u3000\u30005) 종교관련종사자가 기획재정부령으로 정하는 사택을 제공받아 얻는 이익\n\n자. 법령ㆍ조례에 따른 위원회 등의 보수를 받지 아니하는 위원(학술원 및 예술원의 회원을 포함한다) 등이 받는 수당\n\n[전문개정 2009. 12. 31.]\n\n\n\n제13조 삭제 <2009. 12. 31.>\n\n\n\n제2절 과세표준과 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세액계산 통칙 <개정 2009. 12. 31.>\n\n\n\n제14조(과세표준의 계산) ① 거주자의 종합소득 및 퇴직소득에 대한 과세표준은 각각 구분하여 계산한다.\n\n② 

# 3. 제공되는 prompt를 활용하여 답변 생성

In [4]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")


In [5]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever, # database.as_retriever()
    chain_type_kwargs={"prompt": prompt} # 제공되는 prompt를 활용하여 답변 생성
)

In [7]:
ai_message = qa_chain.invoke(query)
ai_message

{'query': '연봉 5000만원인 직장인의 소득세는 얼마인가요?',
 'result': '죄송하지만 주어진 문맥에는 연봉 5000만원인 직장인의 소득세를 계산하는 데 필요한 정보가 포함되어 있지 않습니다. 소득세 계산을 위해 소득세율, 공제 항목 등을 확인해야 합니다. 보다 정확한 계산을 위해서는 현행 소득세율 표와 관련 공제 정보를 참고하시기 바랍니다.'}